dataset: https://www.kaggle.com/datasets/kirilluspredator/ml-trainings-alien-translation

In [2]:
PATH = "/kaggle/input/yandex-ml2-0/ml_trainings.alien_translation/"
OUT_PATH = "/kaggle/working/"

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")

data = {
    "dst": "How fast do you go?",
    "src": "▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?"
}

tokenized_src = tokenizer(data["src"], return_tensors="pt", max_length=256, truncation=True, padding="max_length")
tokenized_dst = tokenizer(data["dst"], return_tensors="pt", max_length=256, truncation=True, padding="max_length")

detokenized_src = tokenizer.decode(tokenized_src.input_ids[0], skip_special_tokens=True)
detokenized_dst = tokenizer.decode(tokenized_dst.input_ids[0], skip_special_tokens=True)

print("Original Source:", data["src"])
print("Tokenized Source IDs:", tokenized_src.input_ids)
print("Detokenized Source:", detokenized_src)

print("\nOriginal Target:", data["dst"])
print("Tokenized Target IDs:", tokenized_dst.input_ids)
print("Detokenized Target:", detokenized_dst)

tokenizer_config.json:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

Original Source: ▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?
Tokenized Source IDs: tensor([[229, 153, 178, 229, 154, 173,  35, 229, 153, 171, 229, 154, 163, 229,
         154, 139, 229, 154, 163, 229, 154, 150,  35, 229, 154, 161, 229, 154,
         176, 229, 154, 150, 229, 154, 163, 229, 153, 174,  35, 229, 154, 182,
         229, 154, 163, 229, 153, 190, 229, 154, 175, 229, 154, 182, 229, 154,
         170, 229, 154, 150,  35, 229, 154, 161, 229, 153, 183, 229, 153, 169,
         229, 154, 154, 229, 153, 169, 229, 153, 171, 229, 154, 174,  66,   1,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
import json
from transformers import AutoTokenizer
from datasets import Dataset

with open(PATH + "train", "r") as f:
    data = [json.loads(line) for line in f]

dataset = [{"src": item["src"], "dst": item["dst"]} for item in data]

def preprocess(batch):
    inputs = [f"translate alien to English: {src}" for src in batch["src"]]
    targets = batch["dst"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length").input_ids

    labels = [[-100 if token == tokenizer.pad_token_id else token for token in label] for label in labels]

    model_inputs["labels"] = labels
    return model_inputs

hf_dataset = Dataset.from_dict({"src": [item["src"] for item in dataset], "dst": [item["dst"] for item in dataset]})

tokenized_dataset = hf_dataset.map(preprocess, batched=True, remove_columns=["src", "dst"])

tokenized_dataset.save_to_disk(OUT_PATH + "tokenized_train_dataset")

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/300000 [00:00<?, ? examples/s]

In [5]:
with open(PATH + "val", "r") as f:
    data_val = [json.loads(line) for line in f]

dataset_val = [{"src": item["src"], "dst": item["dst"]} for item in data_val]

hf_dataset_val = Dataset.from_dict({"src": [item["src"] for item in dataset_val], "dst": [item["dst"] for item in dataset_val]})

tokenized_dataset_val = hf_dataset_val.map(preprocess, batched=True, remove_columns=["src", "dst"])

tokenized_dataset_val.save_to_disk(OUT_PATH + "tokenized_val_dataset")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForSeq2SeqLM
from evaluate import load
import numpy as np

model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
# model = AutoModelForSeq2SeqLM.from_pretrained("./byt5-alien-translation")

bleu_metric = load("bleu")

def compute_bleu(eval_pred):
    predictions, labels = eval_pred
    bleu_metric = load("bleu")
    
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    if predictions.ndim > 1:
        predictions = predictions.argmax(-1)
    
    predictions = predictions.tolist() if hasattr(predictions, 'tolist') else predictions
    labels = labels.tolist() if hasattr(labels, 'tolist') else labels
    
    try:
        decoded_preds = [tokenizer.decode(pred, skip_special_tokens=True) for pred in predictions]
        decoded_labels = [tokenizer.decode(label, skip_special_tokens=True) for label in labels]
    except Exception:
        decoded_preds = [str(pred) for pred in predictions]
        decoded_labels = [str(label) for label in labels]
    
    results = bleu_metric.compute(
        predictions=decoded_preds, 
        references=[[label] for label in decoded_labels]
    )
    
    return {"bleu": results["bleu"]}

training_args = TrainingArguments(
    output_dir="./byt5-alien-translation",  
    eval_strategy="steps",
    eval_steps=500,                         
    learning_rate=5e-5,                    
    per_device_train_batch_size=16,
    per_device_eval_batch_size=63,         
    num_train_epochs=4,                    
    save_strategy="epoch",                 
    save_total_limit=2,                    
    logging_dir="./logs",                  
    report_to="none",                      
    logging_steps=100,
    disable_tqdm=False,                    
    lr_scheduler_type='linear',            
    warmup_steps=1000,                      
)

trainer = Trainer(
    model=model,                         
    args=training_args,                    
    train_dataset=tokenized_dataset,       
    eval_dataset=tokenized_dataset_val,    
    tokenizer=tokenizer, 
    compute_metrics=compute_bleu           
)

trainer.train()
model.save_pretrained("./byt5-alien-translation")
tokenizer.save_pretrained("./byt5-alien-translation")


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Step,Training Loss,Validation Loss,Bleu
500,1.692800,1.341764,0.571940
1000,1.484000,1.281028,0.584291
1500,1.411700,1.277546,0.582426
2000,1.381900,1.269107,0.584469
2500,1.344600,1.252071,0.586476
3000,1.323200,1.257648,0.586653
3500,1.311500,1.255867,0.587843
4000,1.303000,1.258580,0.583987
4500,1.285800,1.251742,0.586934
5000,1.272200,1.255326,0.585500


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("./byt5-alien-translation")
model = AutoModelForSeq2SeqLM.from_pretrained("./byt5-alien-translation")

alien_sentence = "Translate: ◢◧◓▨◨◈◠▦ ▽◫◳▴◎◗▽◧◓◨◎▵?"
input_ids = tokenizer(f"{alien_sentence}", return_tensors="pt").input_ids
output_ids = model.generate(
    input_ids,               
    num_beams=5,             
    max_length=128,          
    early_stopping=True,     
    num_return_sequences=1  
)

translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Translation: {translation}")


In [28]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("./byt5-alien-translation")
model = AutoModelForSeq2SeqLM.from_pretrained("./byt5-alien-translation")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def translate_sentence(sentence):
    input_ids = tokenizer(f"{sentence}", return_tensors="pt").input_ids.to(device)
    
    output_ids = model.generate(
        input_ids,               
        num_beams=5,             
        max_length=512,          
        early_stopping=True,     
        num_return_sequences=1,  
        no_repeat_ngram_size=3,  
        top_k=50,               
        temperature=1.0
    )
    
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

data = []
with open(PATH + "test_no_reference", "r") as f:
    for line in f:
        data.append(json.loads(line)['src'])

In [29]:
from tqdm.auto import tqdm
translated_data=[]

for sentence in tqdm(data):
    translation = translate_sentence('' + sentence)
    
    print(translation)
    
    translated_data.append({
        "src": sentence,
        "dst": translation
    })

with open(OUT_PATH + "translated_output.json", "w") as f:
    for entry in translated_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write("\n")  

print("Translation complete. Results saved to 'translated_output.json'.")


  0%|          | 0/1000 [00:00<?, ?it/s]

Some of these messages weren't friendly, and so many people are not very filming.
And after a message he sent himself to an official motherfucking woman's written in trouble.
She said Brexit waiting for people to lose it.
Sergio Garcia and Webb Simpson made a lot of heroes with Alex Noren in the morning.
She never let them see you're crying.
He said he was too suspicious about someone ever explaining a policeman who escaped from their cops.
And one year after the lieutenant Katalans have been out of Madrid.
He used them about bolivia energy and society with roberto Calzadilla.
Aldi and Lidl said he looks like his supermarket will be more department services.
He started watching a different rota, but he had to see an idiot.
The Elvie/Mother sits inside of his emsirme suture, and he's always sleeping on a bird street, within that shelve pompa out to t
He accepted that we can't see it from ABD.
Wasilla arrested for a report to the family of Palin and Alaska police criminals were attacked.

In [31]:
# !tar -czvf byt5-alien-translation.tar.gz -C /kaggle/working byt5-alien-translation
!rm -rf /kaggle/working byt5-alien-translation/check*


rm: cannot remove '/kaggle/working': Device or resource busy
